# EMET2007 Week 10: Time Series Analysis - Inflation

## Learning Objectives

By the end of this tutorial, you will be able to:
1. Create time series variables from raw data (log-differences for inflation)
2. Plot time series and identify key features
3. Create lagged variables and understand autocorrelation
4. Estimate autoregressive (AR) models
5. Make in-sample predictions and out-of-sample forecasts

---

## Introduction

Today we analyse Australian inflation using data from the Reserve Bank of Australia. The dataset contains the Consumer Price Index (CPI) from 1922:Q2 to 2025:Q1.

**Key concept:** Inflation is calculated as the annualised percentage change in the CPI:

$$\text{inflation}_t = 400 \times (\log(\text{CPI}_t) - \log(\text{CPI}_{t-1}))$$

The 400 converts the quarterly log-difference to an annualised percentage.

---

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.ar_model import AutoReg

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/juergenmeinecke/EMET2007/refs/heads/main/datasets/cpi_aus_2025.csv')

**Run this cell after loading the data** to set up proper time indexing:

In [ ]:
# Convert to datetime and set quarterly index
# df['date'] = pd.to_datetime(df['time'], format='%b-%Y')
# df.index = pd.DatetimeIndex(df.date, name='quarter').to_period('Q')

---

## Part 1: Creating Time Series Variables

### Exercise 1: Calculate Inflation

**Task:** Create two new variables:
1. `logcpi` = log of CPI
2. `infl` = annualised log-difference (inflation rate)

**Syntax hints:**
```python
df['logcpi'] = np.log(df.cpi)
df['infl'] = 400 * df.logcpi.diff()  # .diff() calculates first difference
```

In [ ]:
# Create inflation variable


In [ ]:
# View first and last observations
# df.head(8)
# df.tail(8)


### Exercise 2: Plot the Time Series

**Task:** Create a figure with two subplots:
1. CPI over time
2. Inflation over time

**Questions to consider:**
- What patterns do you see in the CPI?
- What episodes of high inflation can you identify?
- Does inflation appear to be stationary?

In [ ]:
# Create subplot figure
# fig, axs = plt.subplots(2, 1, figsize=(10, 7))

# Plot CPI
# axs[0].plot(df.date, df.cpi)
# axs[0].set_ylabel('CPI')
# axs[0].set_title('Australian CPI')

# Plot inflation
# axs[1].plot(df.date, df.infl)
# axs[1].set_ylabel('Inflation (%)')
# axs[1].set_title('Australian Inflation (annualised)')

# plt.tight_layout()
# plt.show()


---

## Part 2: Understanding Autocorrelation

### Exercise 3: Create Lagged Variables

**Autocorrelation** (serial correlation) measures how a time series is correlated with its own past values.

**Task:** Create lagged versions of CPI and inflation:
```python
df['l1cpi'] = df.cpi.shift(1)   # Lag of order 1
df['l1infl'] = df.infl.shift(1)
```

In [ ]:
# Create lagged variables


In [ ]:
# View to verify
# df[['date', 'cpi', 'l1cpi', 'infl', 'l1infl']].head(8)


### Exercise 4: Calculate Autocorrelations

**Task:** Calculate the correlation between:
1. CPI and its lag (l1cpi)
2. Inflation and its lag (l1infl)

*Why might CPI have very high autocorrelation while inflation has lower autocorrelation?*

In [ ]:
# Autocorrelation of CPI
# df[['cpi', 'l1cpi']].corr()


In [ ]:
# Autocorrelation of inflation
# df[['infl', 'l1infl']].corr()


### Exercise 5: Plot Autocorrelation Functions

The `plot_acf` function shows autocorrelations at multiple lags:

```python
plot_acf(df.variable, missing='drop')
plt.show()
```

In [ ]:
# ACF for CPI


In [ ]:
# ACF for inflation


*What do these plots tell you about the persistence of CPI vs. inflation?*

---

## Part 3: Autoregressive Models

### Exercise 6: Estimate an AR(1) Model

An AR(1) model regresses a variable on its own first lag:

$$\text{infl}_t = \beta_0 + \beta_1 \cdot \text{infl}_{t-1} + u_t$$

**Two ways to estimate:**

Method 1 - Manual (using our familiar OLS):
```python
reg = smf.ols('infl ~ l1infl', data=df, missing='drop').fit()
```

Method 2 - Using AutoReg:
```python
ar1 = AutoReg(df.infl, lags=1, missing='drop').fit()
```

In [ ]:
# Method 1: Manual OLS


In [ ]:
# Method 2: AutoReg


*Note: The coefficients should be identical; minor differences in standard errors are due to different degree-of-freedom adjustments.*

---

## Part 4: Prediction and Forecasting

### Exercise 7: Make Predictions

**Key distinction:**
- **Prediction** (in-sample): Fitted value for a period where we know the actual value
- **Forecast** (out-of-sample): Predicted value for a future period

Our data ends at 2025:Q1. Let's:
1. Predict inflation for 2025:Q1 (using 2024:Q4 data)
2. Forecast inflation for 2025:Q2 (using our prediction for 2025:Q1)

**Manual calculation:**
$$\widehat{\text{infl}}_{2025Q1} = \hat{\beta}_0 + \hat{\beta}_1 \cdot \text{infl}_{2024Q4}$$

**Using predict:**
```python
newdata = {'l1infl': [df.infl['2024Q4'], df.infl['2025Q1']]}
reg.predict(newdata)
```

In [ ]:
# View the coefficient estimates
# reg.params


In [ ]:
# Manual prediction for 2025:Q1
# Manual forecast for 2025:Q2


In [ ]:
# Using predict function


*Compare your prediction for 2025:Q1 to the actual value in the data. How accurate was the model?*

---

## Summary

### Key Concepts

**Time Series Basics:**
- Inflation = annualised log-difference of CPI
- Use `.shift(1)` to create lagged variables
- Use `.diff()` to create first differences

**Autocorrelation:**
- Correlation of a series with its own past values
- High for trending series (CPI), lower for stationary series (inflation)
- `plot_acf()` visualises autocorrelation at multiple lags

**AR(1) Model:**
- Regresses a variable on its own first lag
- Captures persistence in the series
- Useful for forecasting

**Prediction vs. Forecasting:**
- In-sample prediction uses known lagged values
- Out-of-sample forecasting uses predicted values as inputs

---

## Attribution

Data source: Reserve Bank of Australia, [Statistics](https://www.rba.gov.au/statistics).